In [4]:
import json, random, torch, csv
from collections import defaultdict
from functools import partial
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                           TrainingArguments, Trainer)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-3-1b-it"
RAW_FILE = "train_small.jsonl"          
PER_FUNC_MAX = 450
MAX_LENGTH = 1024
LABEL_MAP = {True: " Equivalent", False: " Not equivalent"}

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

def detect_lang(code: str) -> str:
    java_markers = ["public class", "public static void main", "System.out.println",
                     "import java.", "void ", "String[", "int["]
    if any(m in code for m in java_markers):
        return "java"
    if code.count(";") > 3 and "def " not in code:
        return "java"
    return "python"

def strat_key(row):
    l1, l2 = detect_lang(row["func1"]), detect_lang(row["func2"])
    return f"{tuple(sorted([l1, l2]))}_{row['label']}"

def truncate_code(tokenizer, code, max_tokens=PER_FUNC_MAX):
    ids = tokenizer.encode(code, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return code
    return tokenizer.decode(ids[:max_tokens])

def build_text(row, tokenizer, per_func_max=PER_FUNC_MAX):
    f1 = truncate_code(tokenizer, row["func1"], per_func_max)
    f2 = truncate_code(tokenizer, row["func2"], per_func_max)
    return (f"Function A:\n{f1}\n\nFunction B:\n{f2}\n\n"
            f"Question: Do these two functions implement the same functionality?\nAnswer:")



c:\Users\mistr\Documents\project_jarvis\jarvisenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0816 19:33:29.703000 29692 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [6]:
TARGETS = {
    ("java", "python"): {True: float("inf"), False: float("inf")},   # rare — sab rakho
    ("python", "python"): {True: 4500, False: 4500},
    ("java", "java"): {True: 2500, False: 2500},
}

reservoirs = {k: {True: [], False: []} for k in TARGETS}
seen_counts = {k: {True: 0, False: 0} for k in TARGETS}
total_counts_all = {}

with open(RAW_FILE, encoding="utf-8") as f:
    for i, line in enumerate(f):
        row = json.loads(line)
        l1, l2 = detect_lang(row["func1"]), detect_lang(row["func2"])
        ptype = tuple(sorted([l1, l2]))
        label = row["label"]
        total_counts_all[(ptype, label)] = total_counts_all.get((ptype, label), 0) + 1

        target = TARGETS.get(ptype, {}).get(label, 0)
        if target == 0:
            continue
        seen_counts[ptype][label] += 1
        bucket = reservoirs[ptype][label]
        if target == float("inf"):
            bucket.append(row)
        elif len(bucket) < target:
            bucket.append(row)
        else:
            j = random.randint(0, seen_counts[ptype][label] - 1)
            if j < target:
                bucket[j] = row
        if i % 100000 == 0:
            print(f"...{i} lines processed")

print("\nTRUE distribution across FULL file:")
for k, v in sorted(total_counts_all.items()):
    print(k, v)

final_rows = [row for d in reservoirs.values() for rows in d.values() for row in rows]
random.shuffle(final_rows)
save_jsonl(final_rows, "train_subsample.jsonl")
print("Final subsample size:", len(final_rows))

...0 lines processed
...100000 lines processed
...200000 lines processed
...300000 lines processed
...400000 lines processed

TRUE distribution across FULL file:
(('java', 'java'), False) 68388
(('java', 'java'), True) 33203
(('java', 'python'), False) 2174
(('java', 'python'), True) 1314
(('python', 'python'), False) 197053
(('python', 'python'), True) 197868
Final subsample size: 17488


In [7]:
rows = load_jsonl("train_subsample.jsonl")
keys = [strat_key(r) for r in rows]

train_rows, val_rows = train_test_split(rows, test_size=0.1, random_state=42, stratify=keys)

train_keys = set((r["func1"], r["func2"]) for r in train_rows)
val_rows = [r for r in val_rows if (r["func1"], r["func2"]) not in train_keys]   # leakage fix

save_jsonl(train_rows, "train_split.jsonl")
save_jsonl(val_rows, "val_split.jsonl")
print("Train size:", len(train_rows), "| Val size (post leakage-fix):", len(val_rows))

Train size: 15739 | Val size (post leakage-fix): 1743


In [8]:
train_rows = load_jsonl("train_split.jsonl")
val_rows = load_jsonl("val_split.jsonl")

train_final, _ = train_test_split(train_rows, train_size=5500, random_state=42,
                                    stratify=[strat_key(r) for r in train_rows])
val_monitor, _ = train_test_split(val_rows, train_size=min(300, len(val_rows)-1), random_state=42,
                                    stratify=[strat_key(r) for r in val_rows])

save_jsonl(train_final, "train_final.jsonl")
save_jsonl(val_monitor, "val_monitor.jsonl")
print("Train final:", len(train_final), "| Val monitor:", len(val_monitor))

Train final: 5500 | Val monitor: 300


In [13]:

class CodePairDataset(Dataset):
    def __init__(self, rows, tokenizer, per_func_max=PER_FUNC_MAX, max_length=MAX_LENGTH):
        self.rows, self.tokenizer = rows, tokenizer
        self.per_func_max, self.max_length = per_func_max, max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        prompt = build_text(row, self.tokenizer, self.per_func_max)
        completion = LABEL_MAP[row["label"]]
        prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=True)
        completion_ids = self.tokenizer.encode(completion, add_special_tokens=False)
        eos_id = self.tokenizer.eos_token_id
        input_ids = prompt_ids + completion_ids + [eos_id]
        labels = [-100] * len(prompt_ids) + completion_ids + [eos_id]
        if len(input_ids) > self.max_length:
            overflow = len(input_ids) - self.max_length
            input_ids, labels = input_ids[overflow:], labels[overflow:]
        return {"input_ids": input_ids, "labels": labels, "attention_mask": [1]*len(input_ids)}

def collate_fn(batch, tokenizer):
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, attn = [], [], []
    for x in batch:
        p = max_len - len(x["input_ids"])
        input_ids.append(x["input_ids"] + [pad_id]*p)
        labels.append(x["labels"] + [-100]*p)
        attn.append(x["attention_mask"] + [0]*p)
    return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}



In [14]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights: 100%|██████████| 340/340 [00:02<00:00, 131.79it/s]


trainable params: 2,981,888 || all params: 1,002,867,840 || trainable%: 0.2973


In [ ]:

train_final = load_jsonl("train_final.jsonl")
val_monitor = load_jsonl("val_monitor.jsonl")

train_ds = CodePairDataset(train_final, tokenizer)
val_monitor_ds = CodePairDataset(val_monitor, tokenizer)
collate = partial(collate_fn, tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=2,          
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,         
    max_steps=650,
    learning_rate=2e-4,
    fp16=True,                              
    gradient_checkpointing=True,           
    optim="paged_adamw_8bit",              
    dataloader_pin_memory=False,            
    save_strategy="steps",
    save_steps=130,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=130,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_ds,
                   eval_dataset=val_monitor_ds, data_collator=collate)
trainer.train()   

Step,Training Loss,Validation Loss
130,0.203617,0.190472
260,0.141678,0.135431
390,0.120927,0.095225
520,0.100471,0.094722
650,0.143113,0.082227


c:\Users\mistr\Documents\project_jarvis\jarvisenv\lib\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in google/gemma-3-1b-it.
  warnings.warn(
c:\Users\mistr\Documents\project_jarvis\jarvisenv\lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in google/gemma-3-1b-it - will assume that the vocabulary was not modified.
  warnings.warn(


TrainOutput(global_step=650, training_loss=0.1689336244418071, metrics={'train_runtime': 7229.7973, 'train_samples_per_second': 0.719, 'train_steps_per_second': 0.09, 'total_flos': 1.14681855773952e+16, 'train_loss': 0.1689336244418071, 'epoch': 0.9454545454545454})

In [9]:
model.save_pretrained("./my_finetuned_gemma")
tokenizer.save_pretrained("./my_finetuned_gemma")

('./my_finetuned_gemma\\tokenizer_config.json',
 './my_finetuned_gemma\\chat_template.jinja',
 './my_finetuned_gemma\\tokenizer.json')

In [24]:

#del trainer
torch.cuda.empty_cache()
model.eval()
tokenizer.padding_side = "left"   

def predict_batch(model, tokenizer, rows, batch_size=4, max_new_tokens=6):
    preds = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        prompts = [build_text(r, tokenizer) for r in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                            truncation=True, max_length=MAX_LENGTH).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  do_sample=False, pad_token_id=tokenizer.pad_token_id)
        gen_only = out[:, inputs["input_ids"].shape[1]:]
        texts = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
        for t in texts:
            tl = t.strip().lower()
            if "not" in tl:
                preds.append(False)
            elif "equivalent" in tl:
                preds.append(True)
            else:
                preds.append(None)     # malformed generation
        if (i // batch_size) % 20 == 0:
            print(f"...{i}/{len(rows)} done")
    return preds



In [ ]:
val_full = load_jsonl("val_split.jsonl")
val_preds = predict_batch(model, tokenizer, val_full)

malformed = sum(1 for p in val_preds if p is None)
print("Malformed generations:", malformed, "/", len(val_preds))
val_preds_clean = [p if p is not None else False for p in val_preds]
val_labels = [r["label"] for r in val_full]

print("\nOVERALL F1:", f1_score(val_labels, val_preds_clean))

by_type_labels, by_type_preds = defaultdict(list), defaultdict(list)
for r, p in zip(val_full, val_preds_clean):
    ptype = tuple(sorted([detect_lang(r["func1"]), detect_lang(r["func2"])]))
    by_type_labels[ptype].append(r["label"])
    by_type_preds[ptype].append(p)

print("\nBreakdown by pair-type:")
for ptype in by_type_labels:
    f1 = f1_score(by_type_labels[ptype], by_type_preds[ptype])
    print(f"{ptype}: F1={f1:.4f}, n={len(by_type_labels[ptype])}")

...0/1742 done
...80/1742 done
...160/1742 done
...240/1742 done
...320/1742 done
...400/1742 done
...480/1742 done
...560/1742 done
...640/1742 done
...720/1742 done
...800/1742 done
...880/1742 done
...960/1742 done
...1040/1742 done
...1120/1742 done
...1200/1742 done
...1280/1742 done
...1360/1742 done
...1440/1742 done
...1520/1742 done
...1600/1742 done
...1680/1742 done
Malformed generations: 17 / 1742

OVERALL F1: 0.896421845574388

Breakdown by pair-type:
('java', 'python'): F1=0.9120, n=347
('python', 'python'): F1=0.9045, n=900
('java', 'java'): F1=0.8734, n=495


In [ ]:
test_rows = load_jsonl("test.jsonl")            
test_preds = predict_batch(model, tokenizer, test_rows)
test_preds_clean = [p if p is not None else False for p in test_preds]

with open("submission.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "label"])              
    for r, p in zip(test_rows, test_preds_clean):
        writer.writerow([r["id"], p])

print("submission.csv ban gaya —", len(test_preds_clean), "predictions")

...0/5000 done
...80/5000 done
...160/5000 done
...240/5000 done
...320/5000 done
...400/5000 done
...480/5000 done
...560/5000 done
...640/5000 done
...720/5000 done
...800/5000 done
...880/5000 done
...960/5000 done
...1040/5000 done
...1120/5000 done
...1200/5000 done
...1280/5000 done
...1360/5000 done
...1440/5000 done
...1520/5000 done
...1600/5000 done
...1680/5000 done
...1760/5000 done
...1840/5000 done
...1920/5000 done
...2000/5000 done
...2080/5000 done
...2160/5000 done
...2240/5000 done
...2320/5000 done
...2400/5000 done
...2480/5000 done
...2560/5000 done
...2640/5000 done
...2720/5000 done
...2800/5000 done
...2880/5000 done
...2960/5000 done
...3040/5000 done
...3120/5000 done
...3200/5000 done
...3280/5000 done
...3360/5000 done
...3440/5000 done
...3520/5000 done
...3600/5000 done
...3680/5000 done
...3760/5000 done
...3840/5000 done
...3920/5000 done
...4000/5000 done
...4080/5000 done
...4160/5000 done
...4240/5000 done
...4320/5000 done
...4400/5000 done
...4480/

In [16]:
train_split = load_jsonl("train_split.jsonl")
train_final = load_jsonl("train_final.jsonl")
used_keys = set((r["func1"], r["func2"]) for r in train_final)
unused = [r for r in train_split if (r["func1"], r["func2"]) not in used_keys]

def ptype_of(row):
    return tuple(sorted([detect_lang(row["func1"]), detect_lang(row["func2"])]))

jj_unused = [r for r in unused if ptype_of(r) == ("java", "java")]
pp_unused = [r for r in unused if ptype_of(r) == ("python", "python")]
jp_unused = [r for r in unused if ptype_of(r) == ("java", "python")]
print("Unused available — java-java:", len(jj_unused), "| python-python:", len(pp_unused), "| java-python:", len(jp_unused))

random.seed(42)
random.shuffle(jj_unused); random.shuffle(pp_unused); random.shuffle(jp_unused)

increment = jj_unused[:2200] + pp_unused[:300] + jp_unused[:200]
random.shuffle(increment)
save_jsonl(increment, "train_increment.jsonl")
print("Incremental set size:", len(increment))

Unused available — java-java: 2922 | python-python: 5270 | java-python: 2039
Incremental set size: 2700


In [19]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 1. Purana Tokenizer load karein (jo aapne save kiya tha)
tokenizer = AutoTokenizer.from_pretrained("./my_finetuned_gemma")

# 2. Base model wapas 4-bit mein load karein
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it", 
    quantization_config=bnb_config, 
    device_map="auto"
)

# 3. YAHAN HAI MAGIC: Apna Round 1 ka adapter attach karein aur Trainable banayein
model = PeftModel.from_pretrained(
    base_model, 
    "./my_finetuned_gemma",   # Aapka saved folder name
    is_trainable=True         # Ye zaroori hai taaki aage aur train ho sake
)

# 4. Training flags on karein
model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Round 1 ka trained model successfully load ho gaya! Ab trainer2 chala sakte hain.")

Loading weights: 100%|██████████| 340/340 [00:03<00:00, 93.65it/s] 


Round 1 ka trained model successfully load ho gaya! Ab trainer2 chala sakte hain.


In [22]:

increment_rows = load_jsonl("train_increment.jsonl")

inc_ds = CodePairDataset(increment_rows, tokenizer)
val_monitor_ds = CodePairDataset(load_jsonl("val_monitor.jsonl"), tokenizer)
collate = partial(collate_fn, tokenizer=tokenizer)

n_steps = max(50, len(increment_rows) // 8)     # effective batch abhi bhi 8 hai
print("Planned steps:", n_steps, "| Est. time:", round(n_steps * 11.12 / 60, 1), "min")  # ~11 sec/step, verified

continue_args = TrainingArguments(
    output_dir="./checkpoints_v2",
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=n_steps,
    learning_rate=1e-4,                # yeh kam hi rakha — continue-training hai, fresh run nahi
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    dataloader_pin_memory=False,
    save_strategy="steps", save_steps=max(50, n_steps // 3), save_total_limit=2,
    eval_strategy="steps", eval_steps=max(50, n_steps // 3),
    load_best_model_at_end=True, metric_for_best_model="eval_loss",
    logging_steps=10, report_to="none",
)

trainer2 = Trainer(model=model, args=continue_args, train_dataset=inc_ds,
                    eval_dataset=val_monitor_ds, data_collator=collate)
trainer2.train()

Planned steps: 337 | Est. time: 62.5 min


Step,Training Loss,Validation Loss
112,0.110948,0.121715
224,0.085818,0.132445
336,0.086574,0.099117
337,0.086574,0.098722


c:\Users\mistr\Documents\project_jarvis\jarvisenv\lib\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in google/gemma-3-1b-it.
  warnings.warn(
c:\Users\mistr\Documents\project_jarvis\jarvisenv\lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in google/gemma-3-1b-it - will assume that the vocabulary was not modified.
  warnings.warn(
c:\Users\mistr\Documents\project_jarvis\jarvisenv\lib\site-packages\peft\utils\other.py:1496: UserWarning: Unable to fetch remote file due to the following error [Errno 11001] getaddrinfo failed - silently ignoring the lookup for the file config.json in google/gemma-3-1b-it.
  warnings.warn(
c:\Users\mistr\Documents\project_jarvis\jarvisenv\lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in google/gemma-3-1b-it - will assu

TrainOutput(global_step=337, training_loss=0.10883326549912065, metrics={'train_runtime': 3139.7736, 'train_samples_per_second': 0.859, 'train_steps_per_second': 0.107, 'total_flos': 7204557160088064.0, 'train_loss': 0.10883326549912065, 'epoch': 0.9985185185185185})

In [29]:
def predict_batch(model, tokenizer, rows, batch_size=4, max_new_tokens=6):
    preds = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        prompts = [build_text(r, tokenizer) for r in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True,
                            truncation=True, max_length=MAX_LENGTH).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  do_sample=False, pad_token_id=tokenizer.pad_token_id)
        gen_only = out[:, inputs["input_ids"].shape[1]:]
        texts = tokenizer.batch_decode(gen_only, skip_special_tokens=True)
        for t in texts:
            tl = t.strip().lower()
            if "not" in tl:
                preds.append(False)
            elif "equivalent" in tl:
                preds.append(True)
            else:
                preds.append(None)     # malformed generation
        if (i // batch_size) % 20 == 0:
            print(f"...{i}/{len(rows)} done")
    return preds

In [31]:
#del trainer2
torch.cuda.empty_cache()
model.eval()

val_full = load_jsonl("val_split.jsonl")
val_preds = predict_batch(model, tokenizer, val_full)
val_preds_clean = [p if p is not None else False for p in val_preds]
val_labels = [r["label"] for r in val_full]

new_f1 = f1_score(val_labels, val_preds_clean)
print("NEW overall F1:", new_f1, "| OLD:", 0.8964)

by_type_labels, by_type_preds = defaultdict(list), defaultdict(list)
for r, p in zip(val_full, val_preds_clean):
    pt = ptype_of(r)
    by_type_labels[pt].append(r["label"])
    by_type_preds[pt].append(p)
for pt in by_type_labels:
    print(f"{pt}: F1={f1_score(by_type_labels[pt], by_type_preds[pt]):.4f}")

...0/1743 done
...80/1743 done
...160/1743 done
...240/1743 done
...320/1743 done
...400/1743 done
...480/1743 done
...560/1743 done
...640/1743 done
...720/1743 done
...800/1743 done
...880/1743 done
...960/1743 done
...1040/1743 done
...1120/1743 done
...1200/1743 done
...1280/1743 done
...1360/1743 done
...1440/1743 done
...1520/1743 done
...1600/1743 done
...1680/1743 done
NEW overall F1: 0.9162621359223301 | OLD: 0.8964
('java', 'python'): F1=0.9048
('python', 'python'): F1=0.9227
('java', 'java'): F1=0.9109


In [33]:

test_rows = load_jsonl("test.jsonl")
test_preds = predict_batch(model, tokenizer, test_rows)
test_preds_clean = [p if p is not None else False for p in test_preds]
with open("submission_new.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "label"])
    for r, p in zip(test_rows, test_preds_clean):
        writer.writerow([r["id"], p])


...0/5000 done
...80/5000 done
...160/5000 done
...240/5000 done
...320/5000 done
...400/5000 done
...480/5000 done
...560/5000 done
...640/5000 done
...720/5000 done
...800/5000 done
...880/5000 done
...960/5000 done
...1040/5000 done
...1120/5000 done
...1200/5000 done
...1280/5000 done
...1360/5000 done
...1440/5000 done
...1520/5000 done
...1600/5000 done
...1680/5000 done
...1760/5000 done
...1840/5000 done
...1920/5000 done
...2000/5000 done
...2080/5000 done
...2160/5000 done
...2240/5000 done
...2320/5000 done
...2400/5000 done
...2480/5000 done
...2560/5000 done
...2640/5000 done
...2720/5000 done
...2800/5000 done
...2880/5000 done
...2960/5000 done
...3040/5000 done
...3120/5000 done
...3200/5000 done
...3280/5000 done
...3360/5000 done
...3440/5000 done
...3520/5000 done
...3600/5000 done
...3680/5000 done
...3760/5000 done
...3840/5000 done
...3920/5000 done
...4000/5000 done
...4080/5000 done
...4160/5000 done
...4240/5000 done
...4320/5000 done
...4400/5000 done
...4480/

In [34]:
model.save_pretrained("./my_new_finetuned_gemma")
tokenizer.save_pretrained("./my_new_finetuned_gemma")

('./my_new_finetuned_gemma\\tokenizer_config.json',
 './my_new_finetuned_gemma\\chat_template.jinja',
 './my_new_finetuned_gemma\\tokenizer.json')